In [1]:
import h5py
import numpy as np
import pandas as pd
from tensorflow import keras
from sklearn.mixture import GaussianMixture
from scipy.special import erf
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path

In [2]:
base_dir      = Path("/Users/angelaliu/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/Fracture Flow UROP")
transport_dir = base_dir / "Updated Code" / "transport narrowed param ranges"
mat_path = transport_dir / "all_trial_bkt_data.mat"

In [3]:
# Reused from the sibling percentile-BKT notebooks -- keep in sync if those
# notebooks' loader changes.
def load_all_trial_bkt_data(mat_path, ps_indices=None):
    param_sets = {}
    with h5py.File(mat_path, 'r') as f:
        weights = f['weights'][()].T  # (num_param_sets, 3)
        atbd = f['all_trial_bkt_data']  # (1, N) object refs
        indices = ps_indices if ps_indices is not None else range(atbd.shape[1])
        for i in indices:
            ps_grp = f[atbd[0, i]]
            p0 = float(ps_grp['p0'][()].flat[0])
            td = ps_grp['trial_data']
            trial_data = []
            for j in range(td.shape[1]):
                arr = f[td[0, j]][()]
                trial_data.append(arr.flatten() if arr.dtype == np.float64 else np.array([], dtype=np.float64))
            param_sets[i] = {'p0': p0, 'trial_data': trial_data}
    return param_sets, weights


def recover_theta(w):
    disc = 1*w[0] + 2.1*(1 - w[0])           # [1, 2.1]
    b    = 0.01e-3*w[1] + 0.3e-3*(1 - w[1])  # [0.01e-3, 0.3e-3]
    lr   = 0.3*w[2] + 0.5*(1 - w[2])         # [0.3, 0.5]
    return disc, lr, b


def quantile_bkt(bkt, p, Np):
    """pth-percentile breakthrough time for one trial from its raw particle
    arrival times (order statistic), or None if undefined. Copied from
    [per p] distrib of pth percentile bkt.ipynb."""
    k = int(np.floor(p * Np))
    if k < len(bkt):
        return bkt[k]
    return None


def logit(p, eps=1e-4):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

In [4]:
# Same connectivity filter as the sibling notebooks.
P0_THRESHOLD = 0.9
Np = 1000

with h5py.File(mat_path, 'r') as f:
    n_param_sets = f['all_trial_bkt_data'].shape[1]

all_param_sets, weights = load_all_trial_bkt_data(mat_path, range(n_param_sets))
USABLE_PS = sorted(i for i, ps in all_param_sets.items() if ps['p0'] < P0_THRESHOLD)
ps_by_p0 = sorted(USABLE_PS, key=lambda i: all_param_sets[i]['p0'])
print(f"{len(USABLE_PS)} / {n_param_sets} param sets usable (p0 < {P0_THRESHOLD})")

87 / 120 param sets usable (p0 < 0.9)


In [5]:
# Rebuild the PDF/MLE-baseline notebook's precompute stage (2-component
# Gaussian-mixture CDF per (theta,p) group, via sklearn GaussianMixture/EM --
# see "[per p] distrib of pth percentile bkt, PDF fit - GMM and NN.ipynb"
# cell-5 for the full rationale) purely to reconstruct the exact
# normalization (X_mean/X_std/Y_mean/Y_std) that percentile_erf_model.keras
# was trained on -- those stats aren't saved alongside the model, so they
# have to be rederived from the same fit_df the model saw during training.
# No new NN training happens here, just a normalization-matching precompute
# + loading the already-trained model.
p_vals = np.array([0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3,
                    0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95])
MIN_SAMPLES = 20


def erf_mixture_cdf(x, w1, mu1, sigma1, mu2, sigma2):
    z1 = (x - mu1) / (sigma1 * np.sqrt(2))
    z2 = (x - mu2) / (sigma2 * np.sqrt(2))
    return w1 * 0.5 * (1 + erf(z1)) + (1 - w1) * 0.5 * (1 + erf(z2))


def fit_gaussian_mixture_cdf(vals, n_init=5, seed=0):
    x = np.log10(np.asarray(vals, dtype=float))
    x = x[np.isfinite(x)]
    if len(x) < 5:
        return None
    gm = GaussianMixture(n_components=2, random_state=seed, n_init=n_init, reg_covar=1e-6).fit(x.reshape(-1, 1))
    order = np.argsort(gm.means_.flatten())
    w = gm.weights_[order]
    mu = gm.means_.flatten()[order]
    sigma = np.sqrt(gm.covariances_.flatten()[order])
    w1, mu1, sigma1, mu2, sigma2 = w[0], mu[0], sigma[0], mu[1], sigma[1]
    xs = np.sort(x)
    n = len(xs)
    ecdf = (np.arange(1, n + 1) - 0.5) / n
    sse = np.sum((erf_mixture_cdf(xs, w1, mu1, sigma1, mu2, sigma2) - ecdf) ** 2)
    return {'w1': w1, 'mu1': mu1, 'sigma1': sigma1, 'mu2': mu2, 'sigma2': sigma2, 'sse': sse}


records = []
for ps_idx in USABLE_PS:
    ps = all_param_sets[ps_idx]
    disc, lr, b = recover_theta(weights[ps_idx])
    for p in p_vals:
        vals = [quantile_bkt(bkt, p, Np) for bkt in ps['trial_data']]
        vals = np.array([v for v in vals if v is not None])
        if len(vals) < MIN_SAMPLES:
            continue
        fit = fit_gaussian_mixture_cdf(vals)
        if fit is None:
            continue
        records.append({'paramset_idx': ps_idx, 'discspacing': disc, 'length_ratio': lr, 'b': b, 'p': p, **fit})

fit_df = pd.DataFrame(records)
print(f"Rebuilt fit_df: {len(fit_df)} (theta,p) groups across {fit_df['paramset_idx'].nunique()} param sets")

Rebuilt fit_df: 1479 (theta,p) groups across 87 param sets


In [6]:
# Same unconstrained-output-transform pattern as the source notebook (train
# on logit(w1), log(sigma1), log(sigma2); invert only at denormalization
# time) -- reproduced here just to get a working denormalize_Y for the
# loaded model, not to retrain anything.
FEATURE_COLS = ['discspacing', 'length_ratio', 'b']
TARGET_COLS = ['w1', 'mu1', 'sigma1', 'mu2', 'sigma2']


def transform_Y(Y):
    w1, mu1, sigma1, mu2, sigma2 = Y[:, 0], Y[:, 1], Y[:, 2], Y[:, 3], Y[:, 4]
    w1c = np.clip(w1, 1e-4, 1 - 1e-4)
    return np.column_stack([np.log(w1c / (1 - w1c)), mu1, np.log(sigma1), mu2, np.log(sigma2)])


def inverse_transform_Y(Yt):
    lw1, mu1, lsigma1, mu2, lsigma2 = Yt[:, 0], Yt[:, 1], Yt[:, 2], Yt[:, 3], Yt[:, 4]
    w1 = 1 / (1 + np.exp(-lw1))
    return np.column_stack([w1, mu1, np.exp(lsigma1), mu2, np.exp(lsigma2)])


X_all = np.column_stack([fit_df[FEATURE_COLS].values, logit(fit_df['p'].values)])
Y_all = transform_Y(fit_df[TARGET_COLS].values)
X_mean, X_std = X_all.mean(axis=0), X_all.std(axis=0)
Y_mean, Y_std = Y_all.mean(axis=0), Y_all.std(axis=0)


def normalize_X(X):
    return (X - X_mean) / X_std


def denormalize_Y(Y):
    return inverse_transform_Y(Y * Y_std + Y_mean)


model = keras.models.load_model(transport_dir / 'percentile_erf_model.keras', compile=False)
print("Loaded percentile_erf_model.keras (PDF/MLE baseline)")

Loaded percentile_erf_model.keras (PDF/MLE baseline)


In [ ]:
# --- Joint CDF surface: F(theta, p, log t) for a few representative param
# sets ---
#
# For a FIXED theta, the pth-percentile-BKT distribution's CDF (as a
# function of t) is what every diagnostic elsewhere in this project plots as
# a single curve per p (e.g. cell-10 in the PDF/MLE-baseline notebook) --
# here, instead of a handful of separate curves color-coded by p, p itself
# becomes a continuous second axis, and the CDF value becomes a color/height
# -- i.e. a genuine 2D surface F(p, log t) for each theta, rather than a
# family of 1D curves. This directly visualizes how the whole family of
# percentile-BKT distributions varies jointly across p, not just at the ~8
# discrete p_vals used for training/diagnostics elsewhere.
#
# 5 representative param sets spanning the connectivity range (same
# quartile-of-p0 selection pattern used elsewhere in this project).
n_ps = len(ps_by_p0)
REP_PS_IDX = [ps_by_p0[0], ps_by_p0[n_ps // 4], ps_by_p0[n_ps // 2],
              ps_by_p0[3 * n_ps // 4], ps_by_p0[-1]]

# p sampled finely (not just the ~17 training p_vals) -- the whole point is
# a smooth surface across p, which the NN can provide at any p since it was
# trained on logit(p) as a continuous input. Kept inside the training
# p_vals' own range (0.001-0.95) to avoid extrapolating past what the model
# ever saw.
P_GRID = np.linspace(0.02, 0.95, 60)

# Discrete p's for the empirical-data scatter overlay -- same 8-value grid
# used in the sibling notebooks' diagnostic PDFs, so this figure's overlay
# is directly comparable to those.
EMPIRICAL_P_VALS = [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]
CONTOUR_LEVELS = [0.1, 0.3, 0.5, 0.7, 0.9]

# Cache each representative param set's (T_GRID, Z) here, keyed by ps_idx --
# reused by the 3D-plot cell below so it doesn't have to recompute the same
# NN predictions a second time.
SURFACES = {}

pdf_path = transport_dir / '[per p] joint CDF surface F(theta,p,logt), representative param sets.pdf'
with PdfPages(pdf_path) as pdf:
    for ps_idx in REP_PS_IDX:
        ps = all_param_sets[ps_idx]
        disc, lr, b = recover_theta(weights[ps_idx])

        # log-t grid: span the empirical data range across a broad set of
        # percentiles for this param set, plus a margin, so the surface
        # covers everywhere the real data (and the model, close to it) has
        # non-trivial mass.
        all_logvals = []
        for p in EMPIRICAL_P_VALS:
            vals = [quantile_bkt(bkt, p, Np) for bkt in ps['trial_data']]
            vals = np.array([v for v in vals if v is not None])
            if len(vals):
                all_logvals.append(np.log10(vals))
        all_logvals = np.concatenate(all_logvals)
        t_lo, t_hi = all_logvals.min() - 0.3, all_logvals.max() + 0.3
        T_GRID = np.linspace(t_lo, t_hi, 220)

        X_grid = np.array([[disc, lr, b, logit(p)] for p in P_GRID])
        Y_pred = denormalize_Y(model.predict(normalize_X(X_grid), verbose=0))  # (len(P_GRID), 5)

        Z = np.stack([erf_mixture_cdf(T_GRID, *params) for params in Y_pred], axis=0)  # (n_p, n_t)
        SURFACES[ps_idx] = (T_GRID, Z)

        fig, ax = plt.subplots(figsize=(10, 6.5))
        mesh = ax.pcolormesh(T_GRID, P_GRID, Z, shading='auto', cmap='viridis', vmin=0, vmax=1)
        cbar = fig.colorbar(mesh, ax=ax, pad=0.02)
        cbar.set_label('CDF  $F(\\theta, p, \\log t)$')

        cs = ax.contour(T_GRID, P_GRID, Z, levels=CONTOUR_LEVELS, colors='white', linewidths=1.0, alpha=0.8)
        ax.clabel(cs, fmt='%.1f', fontsize=8, colors='white')

        # empirical overlay: raw per-trial percentile-BKT samples at the
        # discrete EMPIRICAL_P_VALS, so the smooth model surface can be
        # visually checked against real data density at those rows.
        for p in EMPIRICAL_P_VALS:
            vals = [quantile_bkt(bkt, p, Np) for bkt in ps['trial_data']]
            vals = np.array([v for v in vals if v is not None])
            if len(vals) < 2:
                continue
            log_vals = np.log10(vals)
            jitter = (np.random.default_rng(0).uniform(-1, 1, len(log_vals)) * 0.006)
            ax.scatter(log_vals, np.full_like(log_vals, p) + jitter, s=3, color='white',
                       alpha=0.25, linewidths=0, zorder=3)

        # Explicit integer-exponent ticks -- ax.get_xticks() on this linear
        # data axis returns fractional positions (e.g. 5.0, 5.25, 5.5, ...)
        # that all round to the same displayed exponent under `.0f`
        # formatting, producing duplicate-looking labels like "10^5 10^5
        # 10^6 10^6". One tick per integer log10(t) avoids that.
        tick_lo, tick_hi = int(np.floor(T_GRID.min())), int(np.ceil(T_GRID.max()))
        ticks = np.arange(tick_lo, tick_hi + 1)
        ax.set_xticks(ticks)
        ax.set_xticklabels([f'$10^{{{t:.0f}}}$' for t in ticks])
        ax.set_xlim(T_GRID.min(), T_GRID.max())
        ax.set_xlabel('breakthrough time (s)')
        ax.set_ylabel('percentile $p$')
        ax.set_title(f'Param set {ps_idx}  disc={disc:.2f}, r={lr:.2f}, b={b:.2e}, p0={ps["p0"]:.3f}\n'
                     f'color/contours = NN-predicted CDF (PDF/MLE-baseline model); '
                     f'white dots = empirical samples at p={EMPIRICAL_P_VALS}')
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

print(f'Saved {pdf_path}  ({len(REP_PS_IDX)} pages)')

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 -- registers the '3d' projection

# --- Same surfaces, rendered as an actual 3D surface plot (log t, p, F)
# instead of a 2D color map -- reuses SURFACES from the cell above (no new
# NN predictions), just a different rendering of the same Z arrays. Viewed
# from above this looks like the pcolormesh version; the 3D rendering makes
# the S-shaped "wall" (the CDF's transition from 0 to 1) visually obvious as
# an actual ramp/cliff in F, and lets you rotate/inspect the p-direction
# curvature (relevant to the monotonicity-in-p question) as a literal
# surface shape rather than only via contour-line spacing.
pdf_path_3d = transport_dir / '[per p] joint CDF surface F(theta,p,logt), representative param sets (3D).pdf'
with PdfPages(pdf_path_3d) as pdf:
    for ps_idx in REP_PS_IDX:
        ps = all_param_sets[ps_idx]
        disc, lr, b = recover_theta(weights[ps_idx])
        T_GRID, Z = SURFACES[ps_idx]

        Tmesh, Pmesh = np.meshgrid(T_GRID, P_GRID)  # both (n_p, n_t), matching Z's shape

        fig = plt.figure(figsize=(10, 7.5))
        ax = fig.add_subplot(111, projection='3d')
        surf = ax.plot_surface(Tmesh, Pmesh, Z, cmap='viridis', vmin=0, vmax=1,
                                rstride=1, cstride=2, linewidth=0, antialiased=True)
        cbar = fig.colorbar(surf, ax=ax, pad=0.08, shrink=0.7)
        cbar.set_label('CDF  $F(\\theta, p, \\log t)$')

        tick_lo, tick_hi = int(np.floor(T_GRID.min())), int(np.ceil(T_GRID.max()))
        ticks = np.arange(tick_lo, tick_hi + 1)
        ax.set_xticks(ticks)
        ax.set_xticklabels([f'$10^{{{t:.0f}}}$' for t in ticks])
        ax.set_xlabel('breakthrough time (s)', labelpad=12)
        ax.set_ylabel('percentile $p$', labelpad=10)
        ax.set_zlabel('CDF $F$', labelpad=6)
        ax.set_title(f'Param set {ps_idx}  disc={disc:.2f}, r={lr:.2f}, b={b:.2e}, p0={ps["p0"]:.3f}')
        ax.view_init(elev=25, azim=-60)
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

print(f'Saved {pdf_path_3d}  ({len(REP_PS_IDX)} pages)')

In [8]:
# --- Conclusion ---
#
# This is a genuinely joint view of F(theta, p, log t), complementing (not
# replacing) the per-p line-plot diagnostics elsewhere: a vertical slice at
# fixed p recovers exactly one of those familiar CDF curves; a horizontal
# slice at fixed log t shows how the CDF value (equivalently, "probability
# that the p-th percentile breakthrough time is already below t") varies
# continuously across p -- which is the part earlier diagnostics never
# visualized directly, since they only ever showed p as a discrete
# color-coded label, not a continuous axis.
#
# Uses the PDF/MLE-baseline model (percentile_erf_model.keras) specifically
# because it's this project's baseline/reference model -- swapping in either
# CDF-fit model (2-erf mixture or skew-logistic) just means loading a
# different saved model + its own denormalize_Y/cdf_fn (see the "[per p]
# distrib of pth percentile bkt, CDF fit - ..." notebooks' cell-5/6 for the
# exact per-model context-building code, or the 3-method comparison PDF's
# build script for a template that loads all three at once).
#
# Inspecting the 5 output pages: well-connected param sets (low p0, e.g.
# param set 8) show clean, smoothly-bowing contours across the full p range
# -- no visible folding. The most marginally-connected param set (112,
# p0=0.899) shows small but real non-monotonic wiggles in the contour lines
# around p~0.4-0.5 (a contour dips left then right again as p increases,
# instead of moving monotonically in one direction) -- a direct visual
# instance of the curve-crossing problem investigated quantitatively in the
# skew-logistic notebook's experimental monotonicity-penalized-training
# section. This surface view makes that failure mode immediately visible as
# a local kink, rather than requiring the pairwise violation-counting used
# there -- worth checking here first when judging whether a monotonicity
# fix (or just more training data for marginal-connectivity param sets, in the
# GaussianMixture fits which naturally get noisier once trial counts drop
# below a few hundred) is worth pursuing further.